# Notebook 01 — Exploration des données brutes

**Objectif de ce notebook :** avant de nettoyer et fusionner quoi que ce soit, on regarde
chaque fichier tel quel pour comprendre :
- ses dimensions (nombre de lignes/colonnes)
- ses formats de dates (elles sont différentes selon les fichiers !)
- son taux de valeurs manquantes
- les problèmes évidents à corriger dans les notebooks suivants

On ne modifie et on ne sauvegarde **rien** ici — ce notebook est purement exploratoire.

ℹ️ **Format des fichiers : Parquet, pas CSV.** Tout le pipeline (01 à 07) lit et écrit des
fichiers `.parquet` plutôt que `.csv` — plus rapides à charger et plus légers sur disque, et le
type des colonnes (ex: `annee_mois` en texte) est préservé automatiquement d'un notebook à
l'autre, sans avoir besoin de le forcer à chaque lecture. Convertis tes 3 fichiers bruts en
`.parquet` (ex: `pd.read_csv(...).to_parquet(...)`) et dépose-les dans `data/raw/` avant de
lancer ce notebook.

**Fichiers attendus dans `data/raw/`** (renomme tes fichiers complets ainsi si besoin) :
- `datashare.parquet` — les 94 caractéristiques de Gu, Kelly & Xiu (2020)
- `StockReturn.parquet` — les rendements mensuels par entreprise
- `MacroData.parquet` — les variables macroéconomiques (Welch & Goyal)


## 0. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")  # pour pouvoir importer config.py, situe a la racine du projet
import config

# Affichage un peu plus lisible dans les sorties pandas
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)


## 1. Chargement des 3 fichiers bruts

In [ ]:
chars = pd.read_parquet(config.FICHIER_CARACTERISTIQUES_BRUT)
returns = pd.read_parquet(config.FICHIER_RETURNS_BRUT)
macro = pd.read_parquet(config.FICHIER_MACRO_BRUT)

print("Chargement termine.")


## 2. Premier coup d'oeil — dimensions et colonnes

In [ ]:
for nom, df in [("Caracteristiques (datashare)", chars),
                ("Rendements (StockReturn)", returns),
                ("Macro (MacroData)", macro)]:
    print(f"--- {nom} ---")
    print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print(f"Colonnes   : {list(df.columns)[:8]}{' ...' if df.shape[1] > 8 else ''}")
    print()


In [ ]:
chars.head()

In [ ]:
returns.head()

In [ ]:
macro.head()

## 3. Caractéristiques (`datashare.parquet`) — exploration détaillée

Cette base contient une ligne par (entreprise, mois). La colonne `permno` identifie
l'entreprise, `DATE` est au format `AAAAMMJJ` (ex: 19570131 = janvier 1957), et
`sic2` donne le secteur d'activité (déjà inclus, pas besoin d'une base séparée).

In [ ]:
print("Types de donnees :")
print(chars.dtypes.value_counts())
print()
print("Nombre d'entreprises distinctes (permno) :", chars['permno'].nunique())
print("Periode couverte (colonne DATE) : de", chars['DATE'].min(), "a", chars['DATE'].max())


In [ ]:
# Pourcentage de valeurs manquantes par colonne, triees de la pire a la meilleure
missing_chars = chars.isna().mean().sort_values(ascending=False) * 100
missing_chars.round(1).head(20)


**A retenir :** dans la vraie base GKX, le taux de valeurs manquantes est très élevé
au début de l'échantillon (années 1950-60) car beaucoup de caractéristiques n'étaient pas
calculables faute de données comptables disponibles. C'est une des raisons pour lesquelles
on envisageait de démarrer l'échantillon plus tard (ex: 1980 ou 1990) plutôt qu'en 1957.

In [ ]:
# Secteur (sic2) : distribution et taux de manquants
print("Valeurs manquantes pour sic2 :", chars['sic2'].isna().mean().round(3) * 100, "%")
chars['sic2'].value_counts(dropna=True).head(10)


## 4. Rendements (`StockReturn.parquet`) — exploration détaillée

Attention : la colonne `RET` de CRSP mélange des nombres (rendements) et des **codes texte**
(ex: `"C"`) qui signifient une donnée manquante ou un delisting. Il faut les repérer avant
de pouvoir faire le moindre calcul dessus.

In [ ]:
print("Dimensions :", returns.shape)
print("Type de la colonne RET :", returns['RET'].dtype)
print("Periode couverte : de", returns['date'].min(), "a", returns['date'].max())
print("Nombre d'entreprises distinctes (PERMNO) :", returns['PERMNO'].nunique())


In [ ]:
# On essaie de convertir RET en nombre ; tout ce qui n'est pas un nombre devient NaN
ret_numerique = pd.to_numeric(returns['RET'], errors='coerce')

# Les valeurs qui existaient (non vides) mais qui ne sont PAS des nombres = codes CRSP a traiter
valeurs_non_numeriques = returns.loc[returns['RET'].notna() & ret_numerique.isna(), 'RET']
print("Valeurs texte trouvees dans RET (codes CRSP a traiter au notebook 03) :")
print(valeurs_non_numeriques.value_counts())


In [ ]:
print(f"% de RET manquant ou non numerique : {ret_numerique.isna().mean() * 100:.1f} %")


## 5. Variables macro (`MacroData.parquet`) — exploration détaillée

Cette base est au format Welch & Goyal : une ligne par mois (`yyyymm`), sans identifiant
d'entreprise. Elle démarre en 1871, bien avant les deux autres fichiers — il faudra la
restreindre à la période utile lors de la fusion.

In [ ]:
print("Dimensions :", macro.shape)
print("Periode couverte (yyyymm) : de", macro['yyyymm'].min(), "a", macro['yyyymm'].max())


In [ ]:
missing_macro = macro.isna().mean().sort_values(ascending=False) * 100
missing_macro.round(1)


**A retenir :** certaines colonnes macro (comme `tbl`, `AAA`, `BAA`) sont vides sur
les premières décennies car ces séries n'existaient pas encore à l'époque. Il faudra
vérifier, une fois restreint à la période utile pour ton projet (ex: à partir de 1980),
si ces colonnes sont bien remplies sur cette période-là.

## 6. Vérification des clés de fusion (dates & identifiants)

C'est l'étape la plus importante de cette exploration : `datashare` et `StockReturn`
utilisent un format `AAAAMMJJ`, alors que `MacroData` utilise `AAAAMM`. Il faut créer
une clé "année-mois" commune aux 3 fichiers avant de pouvoir les fusionner (ce sera fait
au notebook 03, partie A).

In [ ]:
# On cree une colonne "annee_mois" (format AAAAMM, texte) dans chaque fichier
chars['annee_mois'] = chars['DATE'].astype(str).str[:6]
returns['annee_mois'] = returns['date'].astype(str).str[:6]
macro['annee_mois'] = macro['yyyymm'].astype(str)

print(chars[['DATE', 'annee_mois']].head(3))
print(returns[['date', 'annee_mois']].head(3))
print(macro[['yyyymm', 'annee_mois']].head(3))


In [ ]:
# Chevauchement des entreprises entre datashare et StockReturn
permnos_chars = set(chars['permno'].unique())
permnos_returns = set(returns['PERMNO'].unique())
communs = permnos_chars & permnos_returns

print(f"Entreprises dans datashare   : {len(permnos_chars)}")
print(f"Entreprises dans StockReturn : {len(permnos_returns)}")
print(f"Entreprises communes aux deux : {len(communs)}")
print("(Sur un simple echantillon de 10 lignes, ce chevauchement peut etre nul ou tres faible : normal.)")


In [ ]:
# Chevauchement des periodes (annee_mois) entre les 3 fichiers
periodes_chars = set(chars['annee_mois'])
periodes_returns = set(returns['annee_mois'])
periodes_macro = set(macro['annee_mois'])

print("Periodes datashare   :", sorted(periodes_chars)[:5], "...")
print("Periodes StockReturn :", sorted(periodes_returns)[:5], "...")
print("Periodes MacroData   :", sorted(periodes_macro)[:5], "...")
